# 자동차 연비 예측 모델 학습 노트북

Auto MPG 데이터를 사용해 자동차 제원으로 예상 연비(MPG)를 예측하는 회귀 모델을 학습합니다.

학습 모델은 수업에서 다룬 범위에 맞춰 다음 4개만 사용합니다.

- 선형회귀
- 의사결정나무
- 랜덤 포레스트
- KNN 회귀

각 모델의 R², MAE, RMSE를 비교하고 RMSE가 가장 낮은 모델을 최종 모델로 선택합니다.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor

## 1. 데이터 불러오기

Seaborn에서 제공하는 `mpg.csv` 데이터를 사용합니다. 프로젝트의 `data/mpg.csv` 파일이 있으면 로컬 파일을 사용하고, 없으면 공개 URL에서 내려받습니다.

In [ ]:
BASE_DIR = Path.cwd()
DATA_PATH = BASE_DIR / "data" / "mpg.csv"
DATA_URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/mpg.csv"

FEATURES = [
    "cylinders",
    "displacement",
    "horsepower",
    "weight",
    "acceleration",
    "model_year",
]

FEATURE_LABELS = {
    "cylinders": "실린더 수",
    "displacement": "배기량",
    "horsepower": "마력",
    "weight": "무게",
    "acceleration": "가속력",
    "model_year": "연식",
}

In [ ]:
if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
else:
    df = pd.read_csv(DATA_URL)
    DATA_PATH.parent.mkdir(exist_ok=True)
    df.to_csv(DATA_PATH, index=False)

df = df[["mpg", *FEATURES]].dropna()
df.head()

## 2. 데이터 확인

예측 대상은 `mpg`이고, 입력 변수는 실린더 수, 배기량, 마력, 무게, 가속력, 연식입니다.

In [ ]:
print(f"데이터 개수: {len(df)}개")
df.describe().round(2)

In [ ]:
df.isna().sum()

## 3. 학습 데이터와 테스트 데이터 분리

전체 데이터를 학습 데이터 80%, 테스트 데이터 20%로 나눕니다. `random_state=42`를 사용해 실행할 때마다 같은 결과가 나오도록 고정했습니다.

In [ ]:
X = df[FEATURES]
y = df["mpg"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"학습 데이터: {len(X_train)}개")
print(f"테스트 데이터: {len(X_test)}개")

## 4. 회귀 모델 학습

수업에서 배운 회귀 모델들을 학습시키고 성능을 비교합니다. KNN 회귀는 거리 기반 모델이므로 `StandardScaler`로 표준화한 뒤 학습합니다.

In [ ]:
models = {
    "선형회귀": LinearRegression(),
    "의사결정나무": DecisionTreeRegressor(random_state=42, max_depth=5),
    "랜덤 포레스트": RandomForestRegressor(
        random_state=42,
        n_estimators=300,
        max_depth=8,
    ),
    "KNN 회귀": make_pipeline(
        StandardScaler(),
        KNeighborsRegressor(n_neighbors=5),
    ),
}

trained_models = {}
results = []

for model_name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    trained_models[model_name] = model
    results.append(
        {
            "모델": model_name,
            "R2": round(r2_score(y_test, y_pred), 3),
            "MAE": round(mean_absolute_error(y_test, y_pred), 3),
            "RMSE": round(float(rmse), 3),
        }
    )

results_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
results_df

## 5. 최종 모델 선택

RMSE는 예측 오차를 의미하므로 값이 낮을수록 좋은 모델입니다. 따라서 RMSE가 가장 낮은 모델을 최종 모델로 선택합니다.

In [ ]:
best_model_name = results_df.loc[0, "모델"]
best_model = trained_models[best_model_name]

print(f"최종 선택 모델: {best_model_name}")
print(f"RMSE: {results_df.loc[0, 'RMSE']}")
print(f"R2: {results_df.loc[0, 'R2']}")
print(f"MAE: {results_df.loc[0, 'MAE']}")

## 6. 특성 영향 분석

최종 모델이 어떤 입력 변수를 중요하게 사용했는지 확인합니다.

- 랜덤 포레스트나 의사결정나무는 `feature_importances_`를 사용합니다.
- 선형회귀는 회귀 계수의 절댓값을 사용합니다.
- KNN은 모델 구조상 특성 중요도를 직접 제공하지 않아 0으로 표시합니다.

In [ ]:
estimator = best_model
if hasattr(best_model, "named_steps"):
    estimator = list(best_model.named_steps.values())[-1]

if hasattr(estimator, "feature_importances_"):
    importance_values = estimator.feature_importances_
    importance_type = "feature_importances_"
elif hasattr(estimator, "coef_"):
    importance_values = np.abs(estimator.coef_)
    importance_type = "coefficient"
else:
    importance_values = np.zeros(len(FEATURES))
    importance_type = "not_supported"

importance_df = (
    pd.DataFrame(
        {
            "특성": [FEATURE_LABELS[item] for item in FEATURES],
            "중요도": importance_values,
        }
    )
    .sort_values("중요도", ascending=False)
    .reset_index(drop=True)
)

print(f"중요도 계산 방식: {importance_type}")
importance_df

## 7. 예시 데이터 예측

Streamlit 앱에서 사용하는 기본 입력값으로 예상 연비를 예측해 봅니다.

In [ ]:
sample = pd.DataFrame(
    [[4, 140, 90, 2400, 15, 82]],
    columns=FEATURES,
)

prediction = round(float(best_model.predict(sample)[0]), 2)
print(f"예상 연비: {prediction} MPG")

## 8. 결론

본 노트북에서는 Auto MPG 데이터를 사용해 4개의 회귀 모델을 학습하고 성능을 비교했습니다. 최종 Streamlit 앱은 이 학습 흐름과 같은 방식으로 모델을 학습한 뒤, 사용자가 입력한 자동차 제원에 대한 예상 연비를 출력합니다.